# Class 9. Exam prep

## Plan: 

<h2>

1) Exam demo <br>

</h2>

# <center> Advanced Python Exam 2026 (demo)</center>

## Rules

1) The exam is open-book, thus you're allowed to have written notes, but you are still **not** allowed to carry and/or use flash drives and other\
electronic tech. and/or software of any kind to assist you

2) The exam will take 2 hours 40 minutes

3) You can ask any questions, but no answer is promised unless it is to a question related to the task description

4) Import the neccessary modules in the cell below the `Imports` label

If you're going to be crying please do it silently so as not to disturb the rest of the students

## Imports

In [1]:
import pandas as pd
import psycopg2

## Task 1 (50 pts). Average Selling Price

### Task description

Table: `Prices`


| Column Name   | Type    |
|---------------|---------|
| product_id    | int     |
| start_date    | date    |
| end_date      | date    |
| price         | int     |

(product_id, start_date, end_date) is the primary key (combination of columns with unique values) for this table.\
Each row of this table indicates the price of the product_id in the period from start_date to end_date.\
For each product_id there will be no two overlapping periods. That means there will be no two intersecting periods for the same product_id.
 

Table: `UnitsSold`


| Column Name   | Type    |
|---------------|---------|
| product_id    | int     |
| purchase_date | date    |
| units         | int     |

This table may contain duplicate rows.
Each row of this table indicates the date, units, and product_id of each product sold. 
 

Write a solution to find the average selling price for each product. average_price should be rounded to 2 decimal places. If a product does not have any sold units, its average selling price is assumed to be 0.

Return the result table in any order.

The result format is in the following example.

 

Example 1:

Input: 
Prices table:

| product_id | start_date | end_date   | price  |
|------------|------------|------------|--------|
| 1          | 2019-02-17 | 2019-02-28 | 5      |
| 1          | 2019-03-01 | 2019-03-22 | 20     |
| 2          | 2019-02-01 | 2019-02-20 | 15     |
| 2          | 2019-02-21 | 2019-03-31 | 30     |

UnitsSold table:

| product_id | purchase_date | units |
|------------|---------------|-------|
| 1          | 2019-02-25    | 100   |
| 1          | 2019-03-01    | 15    |
| 2          | 2019-02-10    | 200   |
| 2          | 2019-03-22    | 30    |

Output: 

| product_id | average_price |
|------------|---------------|
| 1          | 6.96          |
| 2          | 16.96         |

Explanation: 
Average selling price = Total Price of Product / Number of products sold.\
Average selling price for product 1 = ((100 * 5) + (15 * 20)) / 115 = 6.96\
Average selling price for product 2 = ((200 * 15) + (30 * 30)) / 230 = 16.96

In [2]:
def task_1(prices: pd.DataFrame, units_sold: pd.DataFrame) -> pd.DataFrame:
    
    df = prices.merge(units_sold, on='product_id', how='left')
    df = df[(df['start_date'] <= df['purchase_date']) & (df['purchase_date'] <= df['end_date'])].fillna(0)
    df['profit'] = df['price'] * df['units']
    df = df.groupby('product_id').agg({'profit' : 'sum', 'units' : 'sum'}).reset_index()
    df['average_price'] = round(df['profit'] / df['units'], 2) if len(df['profit']) and len(df['units']) else 0
    df = df.merge(prices[['product_id']].drop_duplicates('product_id'), on='product_id', how='right').fillna(0)
    return df[['product_id', 'average_price']]

### Tests

In [146]:
import os
from warnings import filterwarnings

filterwarnings('ignore')

test_dir = 'task_1'
for idx in range(1, 101):
    prices = pd.read_csv(os.path.join(test_dir, f'test_{idx}_prices.csv'))
    units_sold = pd.read_csv(os.path.join(test_dir, f'test_{idx}_units_sold.csv'))
    expected = pd.read_csv(os.path.join(test_dir, f'test_{idx}_expected.csv'))
    try:
        out = task_1(prices=prices, units_sold=units_sold)
        assert_frame_equal(out, expected, check_dtype=False)
    except Exception as e:
        print(f'\rFailed test {idx}:\nprices:\n{prices}\nunits_sold:\n{units_sold}\nError:\n{e}\nGot:\n{out}\nExpected:\n{expected}', end='')
        break
    else:
        print(f'\rCompleted test {idx}', end='')

else:
    print('\rAll tests passed!   ')

All tests passed!   


### SQL implementation

In [5]:
import pandas as pd
import psycopg2

def prob_1(prices: pd.DataFrame,
           units_sold: pd.DataFrame
          ) -> pd.DataFrame:

    with open('pwd.txt', 'r') as file:
        pwd = file.read().strip('\n')

    def connect(
        pwd: str,
        db_name: str = 'postgres',
        usr_nm: str = 'postgres',
        port: str = '5432',
        host: str = 'localhost'
        ) -> tuple[psycopg2.extensions.connection, psycopg2.extensions.cursor]:
        '''
        Не надо писать один код два раза- будьте ленивыми 
        ©Сунь-Цзы
        '''
        conn = psycopg2.connect(
            database=db_name,
            user=usr_nm,
            password=pwd,
            host=host,
            port=port
        )
        return conn

    def to_sql(
        df: pd.DataFrame,
        conn: psycopg2.extensions.connection,
        cur: psycopg2.extensions.cursor | None = None,
        table_nm: str = 'new_table',
        mapping: dict | None = None
    ) -> tuple[psycopg2.extensions.connection, psycopg2.extensions.cursor]:

        assert isinstance(mapping, dict), 'Wrong type of argument passed'
        if cur is None:
            cur = conn.cursor()

        dtypes = df.dtypes.reset_index()
        dtypes[2] = dtypes.iloc[:, -1].apply(lambda x: mapping[str(x)])

        cols = '\n'.join([
            f'{col} {dtype},'
            for col, dtype in zip(df.columns, dtypes.iloc[:, -1])
        ]).strip(',')

        cur.execute(f'DROP TABLE IF EXISTS {table_nm}')
        conn.commit()

        cur.execute(f'''
            CREATE TABLE IF NOT EXISTS {table_nm} (
                {cols}
            );
        ''')
        conn.commit()

        for idx, row in df.iterrows():
            cur.execute(
                f'INSERT INTO {table_nm} ({", ".join(row.index)}) VALUES ({", ".join(["%s"] * len(row))})',
                row.to_list()
            )
            conn.commit()

        return conn, cur


    prices = prices.copy()
    units_sold = units_sold.copy()
    prices['start_date'] = pd.to_datetime(prices['start_date'])
    prices['end_date'] = pd.to_datetime(prices['end_date'])
    units_sold['purchase_date'] = pd.to_datetime(units_sold['purchase_date'])


    with connect(pwd) as conn:

        cur = conn.cursor()

        dtype_mapping = {
            'int64': 'INT',
            'float64': 'FLOAT',
            'object': 'TEXT',
            'datetime64[ns]': 'DATE'
        }
    
    
        conn, cur = to_sql(prices, conn, cur, 'prices', dtype_mapping)
    
        conn, cur = to_sql(units_sold, conn, cur, 'units_sold', dtype_mapping)
    
        query = """
            SELECT 
                p.product_id,
                COALESCE(
                    ROUND(
                        SUM(u.units * p.price)::numeric / NULLIF(SUM(u.units), 0),
                        2
                    ),
                    0
                ) AS average_price
            FROM prices p
            LEFT JOIN units_sold u
                ON p.product_id = u.product_id
                AND u.purchase_date BETWEEN p.start_date AND p.end_date
            GROUP BY p.product_id
            ORDER BY p.product_id;   -- order is not required but keeps output consistent
        """
        result_df = pd.read_sql_query(query, conn)
    
        cur.close()
        
    return result_df

In [6]:
import pandas as pd

#Тестики, чтобы проверить, что я не накосячил (по крайней мере не так, чтобы было заметно)

prices_data = {
    'product_id': [1, 1, 2, 2],
    'start_date': ['2019-02-17', '2019-03-01', '2019-02-01', '2019-02-21'],
    'end_date': ['2019-02-28', '2019-03-22', '2019-02-20', '2019-03-31'],
    'price': [5, 20, 15, 30]
}
prices = pd.DataFrame(prices_data)

units_sold_data = {
    'product_id': [1, 1, 2, 2],
    'purchase_date': ['2019-02-25', '2019-03-01', '2019-02-10', '2019-03-22'],
    'units': [100, 15, 200, 30]
}
units_sold = pd.DataFrame(units_sold_data)

result = prob_1(prices, units_sold)
print(result)

   product_id  average_price
0           1           6.96
1           2          16.96


C:\Users\Tecno\AppData\Local\Temp\ipykernel_12132\491600532.py:111: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  result_df = pd.read_sql_query(query, conn)


## Task 2 (50 pts)

### Task description

Table: `Views`


| Column Name   | Type    |
|---------------|---------|
| article_id    | int     |
| author_id     | int     |
| viewer_id     | int     |
| view_date     | date    |

There is no primary key (column with unique values) for this table, the table may have duplicate rows.\
Each row of this table indicates that some viewer viewed an article (written by some author) on some date.\
Note that equal author_id and viewer_id indicate the same person.
 

Write a solution to find all the authors that viewed at least one of their own articles.

Return the result table sorted by id in ascending order.

The result format is in the following example.

 

Example 1:

Input: 
Views table:

| article_id | author_id | viewer_id | view_date  |
|------------|-----------|-----------|------------|
| 1          | 3         | 5         | 2019-08-01 |
| 1          | 3         | 6         | 2019-08-02 |
| 2          | 7         | 7         | 2019-08-01 |
| 2          | 7         | 6         | 2019-08-02 |
| 4          | 7         | 1         | 2019-07-22 |
| 3          | 4         | 4         | 2019-07-21 |
| 3          | 4         | 4         | 2019-07-21 |

Output: 

| id   |
|------|
| 4    |
| 7    |


In [136]:
def task_2(views: pd.DataFrame) -> pd.DataFrame:
    return views[views['author_id'] == views['viewer_id']][['author_id']].rename(columns={'author_id' : 'id'}).drop_duplicates().sort_values('id').reset_index(drop=True)

### Tests

In [145]:
import os

folder = 'task_2'

for idx in range(1, 101):
    views, expected = pd.read_csv(os.path.join(folder, f'views_{idx}.csv'), index_col=0), pd.read_csv(os.path.join(folder, f'expected_{idx}.csv'), index_col=0)
    try:
        out = task_2(views)
        assert_frame_equal(out, expected, check_dtype=False, check_index_type=False)
    except Exception as e:
        print(f'\rFailed test {idx}:\nviews:\n{views}\n\nError:\n{e}\n\nGot:\n{out}\n\nExpected:\n{expected}', end='')
        break
    else:
        print(f'\rCompleted test {idx}', end='')

else:
    print('\rAll tests passed!   ')

All tests passed!   


### SQL implementation

In [19]:
def prob_2(table: pd.DataFrame | None = None) -> pd.DataFrame:

    with open('pwd.txt', 'r') as file:
        pwd = file.read().strip('\n')

    def connect(pwd: str,
                db_name: str = 'postgres',
                usr_nm: str = 'postgres',
                port: str = '5432',
                host: str = 'localhost'
               ) -> tuple[psycopg2.extensions.connection, psycopg2.extensions.cursor]:
        
        conn = psycopg2.connect(database = db_name,
                                user = usr_nm,
                                password= pwd,
                                host= host,
                                port= port)

        return conn


    def to_sql(df: pd.DataFrame,
               conn: psycopg2.extensions.connection,
               cur: psycopg2.extensions.cursor | None = None,
               table_nm: str = 'new_table',
               mapping: dict | None = None
              ) -> tuple[psycopg2.extensions.connection, psycopg2.extensions.cursor]:

        assert isinstance(mapping, dict), 'Wrong type of argument passed'
        
        if cur is None:
            cur = conn.cursor()

        dtypes = df.dtypes.reset_index()
        # print(dtypes)
        dtypes[2] = dtypes.iloc[:, -1].apply(lambda x: mapping[str(x)])
        # print(dtypes)

        cols = '\n'.join([f'{col} {dtype},' for col, dtype in zip(df.columns, dtypes.iloc[:, -1])]).strip(',')
        # print(cols)
    
        query = f'DROP TABLE IF EXISTS {table_nm}'
        
        cur.execute(query)

        conn.commit()
        
        query = f'''CREATE TABLE IF NOT EXISTS {table_nm} (
            {cols}
        );'''

        # print(query)
        
        cur.execute(query)

        conn.commit()


        for idx, row in df.iterrows():
            cur.execute(f'INSERT INTO {table_nm} ({', '.join(row.index)}) VALUES ({', '.join(['%s']*len(row))})', row.to_list())
            conn.commit()

        return conn, cur

    with connect(pwd) as conn:

        cur = conn.cursor()

        conn, cur = to_sql(table, conn, cur, mapping = {'object' : 'VARCHAR(20)', 'float64' : "REAL", 'int32' : 'INTEGER'})
    
        query = f'''SELECT * FROM (
                    SELECT DISTINCT author_id as id FROM new_table WHERE author_id = viewer_id
                ) ORDER BY id ASC
                '''
        
        cur.execute(query)
        data = cur.fetchall()
        # print(data)
        data = sum(data, ())
        # print(data)
        df = pd.DataFrame({cur.description[0][0] : data})
        return df

In [20]:
import numpy as np

np.random.seed(42)

df = pd.DataFrame({'author_id' : np.random.randint(1, 10, 100),
                   'viewer_id' : np.random.randint(1, 10, 100)})

prob_2(df)

[(1,), (2,), (5,), (7,), (8,), (9,)]
(1, 2, 5, 7, 8, 9)
   id
0   1
1   2
2   5
3   7
4   8
5   9


In [6]:
(1,2) + (3,)

(1, 2, 3)